In [1]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 37.3 MB/s eta 0:00:00


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import pandas as pd

In [7]:
from google.colab import files

uploaded = files.upload()

Saving papers.json to papers.json


In [8]:
import pandas as pd

df = pd.read_json("papers.json")

print(f"Loaded {len(df)} papers.")

Loaded 100 papers.


In [9]:
documents = []

for _, paper in df.iterrows():

    searchable_text = f"""
Title:
{paper['title']}

Abstract:
{paper['abstract']}
"""

    documents.append(searchable_text)

print(f"Prepared {len(documents)} searchable documents.")

Prepared 100 searchable documents.


In [10]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [11]:
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True
)

print("Embedding generation finished.")
print("Embedding matrix shape:", embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding generation finished.
Embedding matrix shape: (100, 384)


In [12]:
vector_dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(
    vector_dimension
)

faiss_index.add(
    np.array(embeddings).astype("float32")
)

print(
    f"Stored {faiss_index.ntotal} vectors in FAISS."
)

Stored 100 vectors in FAISS.


In [13]:
query = """
multimodal deepfake detection using transformers
"""

query_embedding = embedding_model.encode(
    [query]
)

In [14]:
distances, indices = faiss_index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

print("Semantic retrieval completed.")

Semantic retrieval completed.


In [15]:
for rank, idx in enumerate(indices[0], start=1):

    print("=" * 90)

    print(f"Result {rank}")

    print()

    print("Title:")
    print(df.iloc[idx]["title"])

    print()

    print("Published:")
    print(df.iloc[idx]["published"])

    print()

    print("Categories:")
    print(df.iloc[idx]["categories"])

Result 1

Title:
Integrating Audio-Visual Features for Multimodal Deepfake Detection

Published:
2023-10-05

Categories:
['cs.CV']
Result 2

Title:
MIS-AVoiDD: Modality Invariant and Specific Representation for Audio-Visual Deepfake Detection

Published:
2023-10-03

Categories:
['cs.CV', 'cs.AI', 'cs.LG']
Result 3

Title:
Leveraging large multimodal models for audio-video deepfake detection: a pilot study

Published:
2026-02-25

Categories:
['cs.SD', 'cs.CV']
Result 4

Title:
Deepfake Synthesis vs. Detection: An Uneven Contest

Published:
2026-02-08

Categories:
['cs.CV']
Result 5

Title:
WWW: Where, Which and Whatever Enhancing Interpretability in Multimodal Deepfake Detection

Published:
2024-08-06

Categories:
['cs.CV']


In [16]:
faiss.write_index(
    faiss_index,
    "researchforge_faiss.index"
)

print("FAISS index saved successfully.")

FAISS index saved successfully.


In [19]:
np.save(
    "researchforge_embeddings.npy",
    embeddings
)

print("Embeddings saved successfully.")

Embeddings saved successfully.


In [20]:
from google.colab import files

files.download("researchforge_faiss.index")
files.download("researchforge_embeddings.npy")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>